In [12]:
from functools import reduce

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import pickle

In [13]:
preds_path_ls = [
    ('correct',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\correct.bin'),
    ('LoRA',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\LoRA-cfg18-CR.bin'),
    ('DoRA',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\DoRA-cfg18-CR.bin'),
    ('VeRA',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\VeRA-cfg1-CR.bin'),
    ('GSOFT',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\GSOFT-cfg3-CR.bin'),
]

In [46]:
preds_dfs = []

for run_name, preds_path in preds_path_ls:
    with open(preds_path, 'rb') as f:
        preds_df = pickle.load(file=f)

        preds_df = preds_df[preds_df['task'] != 'SIQA']

        preds_dfs.append(preds_df)

In [261]:
preds_dfs[1]

,text,text_wa_answer,task,model_pred
0,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,BoolQ,{'generated_text': ': False'}
1,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,BoolQ,{'generated_text': ': False'}
2,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,BoolQ,{'generated_text': ': False'}
3,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,BoolQ,{'generated_text': ': True'}
4,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,BoolQ,{'generated_text': ': False'}
...,...,...,...,...
19735,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,OBQA,{'generated_text': ' D: placing seed'}
19736,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,OBQA,{'generated_text': ' D: producer.'}
19737,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,OBQA,{'generated_text': ' A: equally distant'}
19738,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,OBQA,{'generated_text': ' C: Water changing'}


In [48]:
tasks = preds_dfs[0].task.unique()

for task in tasks:
    has_dups = np.any(preds_dfs[0][preds_dfs[0]['task'] == task]['text'].value_counts() > 1)
    print(f"{task}: {has_dups=}")

BoolQ: has_dups=False
PIQA: has_dups=False
hellaswag: has_dups=False
winogrande: has_dups=False
ARC-E: has_dups=False
ARC-C: has_dups=False
OBQA: has_dups=False


In [49]:
preds_dfs[0]['text_wa_answer'].value_counts()[(preds_dfs[0]['text_wa_answer'].value_counts() > 1)]

Series([], Name: count, dtype: int64)

In [253]:
import string
import re

punct_trans = str.maketrans(dict.fromkeys(string.punctuation))

def BoolQ_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

def SIQA_process(pred):
    raise NotADirectoryError()

PIQA_pattern = re.compile('Solution[1,2]')
def PIQA_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip() if pred else ''
    pred = re.match(PIQA_pattern, pred)
    pred = pred.group(0) if pred else 'None'

    return pred

hellaswag_pattern = re.compile('(Ending)?[0,1,2,3]')
def hellaswag_process(pred):
    pred = pred['generated_text']
    pred_0 = pred

    pred = pred.translate(punct_trans).strip()
    pred = pred.split()[0] if pred else ''
    pred = re.match(hellaswag_pattern, pred)
    if pred:
        pred = pred.group(0)
        pred = f'Ending{pred[-1]}'
    else:
        pred = pred_0

    return pred

winogrande_pattern = re.compile('Option[1,2]')
def winogrande_process(pred):
    pred = pred['generated_text']
    pred_0 = pred

    pred = pred.translate(punct_trans).strip()
    pred = pred.split()[0] if pred else ''
    pred = re.match(winogrande_pattern, pred)
    if pred:
        pred = pred.group(0)
        pred = f'Option{pred[-1]}'
    else:
        pred = pred_0

    return pred

def ARC_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

def OBQA_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

In [271]:
df_merged = pd.merge(
    left=preds_dfs[0],
    right=preds_dfs[1],
    how='inner',
    on=['text', 'text_wa_answer', 'task'],
    validate='1:1'
)

In [272]:
tasks

array(['BoolQ', 'PIQA', 'hellaswag', 'winogrande', 'ARC-E', 'ARC-C',
       'OBQA'], dtype=object)

In [273]:
task_postprocessors = {
    'BoolQ': BoolQ_process,
    'PIQA': PIQA_process,
    'hellaswag': hellaswag_process,
    'winogrande': winogrande_process,
    'ARC-E': ARC_process,
    'ARC-C': ARC_process,
    'OBQA': OBQA_process
}

for task, processor in task_postprocessors.items():
    df_merged.loc[df_merged['task'] == task, 'model_pred'] = df_merged.loc[df_merged['task'] == task, 'model_pred'].apply(processor)

df_merged.head()

,text,text_wa_answer,correct_answer,task,model_pred
0,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,False,BoolQ,False
1,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False
2,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False
3,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,True
4,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False


In [274]:
df_merged.loc[:, 'is_correct'] = (df_merged.loc[:, 'correct_answer'] == df_merged.loc[:, 'model_pred'])

df_merged.head()

,text,text_wa_answer,correct_answer,task,model_pred,is_correct
0,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,False,BoolQ,False,True
1,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,False
2,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,False
3,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,True,True
4,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,False


In [276]:
df_merged[['task', 'is_correct']].groupby(by='task', as_index=False)['is_correct'].mean()

,task,is_correct
0,ARC-C,0.822742
1,ARC-E,0.924561
2,BoolQ,0.650153
3,OBQA,0.796000
4,PIQA,0.764962
5,hellaswag,0.661522
6,winogrande,0.613260
